# Public datasets, provenance, and analytical scope

Switch among checksum-verified UCI datasets, retrieve official macroeconomic series when explicitly enabled, or validate a local CFPB complaint extract. Network access remains opt-in so the notebook executes offline.

All data are generated locally unless this notebook explicitly calls a reviewed adapter. Results are educational and require independent validation before any real use.

In [ ]:
import os

from creditriskbook.data.datasets import available_datasets, load_dataset

dataset_key = os.getenv("BOOK_DATASET", "synthetic_retail")
if dataset_key not in available_datasets():
    raise ValueError(f"Unknown BOOK_DATASET={dataset_key!r}")
bundle = load_dataset(dataset_key, n_rows=2_000, seed=808)
print({
    "key": bundle.key,
    "shape": bundle.frame.shape,
    "target": bundle.target,
    "licence": bundle.licence,
    "attribution": bundle.attribution,
    "limitations": bundle.limitations,
})
assert len(bundle.source_sha256) == 64

The checksum-verified UCI keys are `uci_south_german`, `uci_statlog_german`, `uci_taiwan_credit_card`, `uci_credit_approval`, `uci_australian_credit_approval`, `uci_polish_bankruptcy`, `uci_taiwan_bankruptcy`, and `uci_bank_marketing`. Approval, bankruptcy, deposit-subscription and default targets retain their source meanings. The optional Kaggle adapter requires a file obtained by the student after reviewing the current dataset-specific terms; the repository does not bundle competition files.

In [ ]:
if os.getenv("BOOK_LIVE_PUBLIC") == "1":
    from creditriskbook.data import load_world_bank_wdi

    macro = load_world_bank_wdi(
        countries=("GRC", "DEU"),
        indicators=("NY.GDP.MKTP.KD.ZG", "SL.UEM.TOTL.ZS"),
        start_year=2020,
        end_year=2025,
    )
    print(macro.frame.tail(8).to_string(index=False))
    print(macro.provenance)
else:
    print("World Bank API example skipped; set BOOK_LIVE_PUBLIC=1 to run it.")

In [ ]:
from pathlib import Path

cfpb_path = os.getenv("BOOK_CFPB_CSV")
if cfpb_path:
    from creditriskbook.data import load_cfpb_complaint_extract

    complaints = load_cfpb_complaint_extract(Path(cfpb_path), include_narratives=False)
    print(complaints.frame.shape, complaints.limitations)
else:
    print("CFPB example skipped; set BOOK_CFPB_CSV to an official local extract.")